In [1]:
!pip install pyswarm

  Preparing metadata (setup.py) ... done
  Created wheel for pyswarm: filename=pyswarm-0.6-py3-none-any.whl size=4463 sha256=3b18b60612b8651e1e7038e884a0fae65e7a873fa33efcbc6af0a6c9ab8fc06b
  Stored in directory: /root/.cache/pip/wheels/93/15/89/3970ef96abd6123028010a90f007c4e6a2bed700db0aa2d36a
Successfully built pyswarm


In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from pyswarm import pso
from google.colab import files

In [ ]:
# Upload dataset manually in Google Colab
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename) # Changed to read_excel

# Separate features and target
X = df.drop(columns=['target']) # Assuming 'target' is the label column
y = df['target']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the dataset
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define ANN model function
def create_ann_model(hidden_layer_1, hidden_layer_2, learning_rate):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train.shape[1],)), # Explicit input layer
        tf.keras.layers.Dense(int(hidden_layer_1), activation='relu'),
        tf.keras.layers.Dense(int(hidden_layer_2), activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# Define fitness function for PSO
def fitness_function(params):
    hidden_layer_1, hidden_layer_2, learning_rate = params
    model = create_ann_model(hidden_layer_1, hidden_layer_2, learning_rate)
    model.fit(X_train, y_train, epochs=10, batch_size=16, verbose=0, validation_split=0.2)
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    return 1 - accuracy # Minimize error

# Define PSO bounds
lb = [4, 4, 0.0001] # Lower bounds for neurons in layers and learning rate
ub = [128, 128, 0.1] # Upper bounds

# Run PSO
best_params, _ = pso(fitness_function, lb, ub, swarmsize=10, maxiter=5)

# Train final model with best parameters
best_model = create_ann_model(*best_params)
best_model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=1, validation_split=0.2)

# Evaluate model
loss, accuracy = best_model.evaluate(X_test, y_test)
print(f"Final Model Accuracy: {accuracy * 100:.2f}%")

# Take dynamic input from user
print("Enter new patient data:")
age = int(input("Enter age: "))
sex = int(input("Enter sex (1 = Male, 0 = Female): "))
cp = int(input("Enter chest pain type (0-3): "))
trestbps = int(input("Enter resting blood pressure: "))
chol = int(input("Enter cholesterol level: "))
fbs = int(input("Fasting blood sugar > 120 mg/dl (1 = Yes, 0 = No): "))
restecg = int(input("Resting ECG results (0-2): "))
thalach = int(input("Max heart rate achieved: "))
exang = int(input("Exercise-induced angina (1 = Yes, 0 = No): "))
oldpeak = float(input("ST depression induced by exercise: "))
slope = int(input("Slope of peak exercise ST segment (0-2): "))
ca = int(input("Number of major vessels (0-3): "))
thal = int(input("Thalassemia type (1-3): "))

new_patient_data = np.array([[age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal]])
new_patient_data = scaler.transform(new_patient_data) # Normalize input

# Predict for new patient
y_pred = best_model.predict(new_patient_data)
prediction = "Heart Disease Detected" if y_pred[0][0] > 0.5 else "No Heart Disease"
print(f"Prediction: {prediction}")

Saving TEST#2.xlsx to TEST#2.xlsx
Stopping search: maximum iterations reached --> 5
Epoch 1/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6899 - loss: 0.5462 - val_accuracy: 0.8478 - val_loss: 0.3574
Epoch 2/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8358 - loss: 0.3640 - val_accuracy: 0.8859 - val_loss: 0.2969
Epoch 3/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8800 - loss: 0.2670 - val_accuracy: 0.8804 - val_loss: 0.2617
Epoch 4/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9105 - loss: 0.2348 - val_accuracy: 0.9049 - val_loss: 0.2249
Epoch 5/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9405 - loss: 0.1792 - val_accuracy: 0.9348 - val_loss: 0.2094
Epoch 6/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9510 - loss: 0.1415 - val_accuracy: 0.9457 - val_loss: 0.2089
Epoch 7/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9583 - loss: 0.1253 - val_accuracy: 0.9402 - val_loss: 0.1788
Epoch 8/50
92/92 ━━━━━━━━━━━━━━━━